In [1]:
# Install the core library
!pip install torch_geometric

# Install the versions strictly matching your PyTorch version
!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv pyg_lib -f https://data.pyg.org/whl/torch-{version}.html

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.0 MB/s eta 0:00:0000:01
Looking in links: https://data.pyg.org/whl/torch-{version}.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.0/210.0 kB 11.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
ERROR: Could not find a version that satisfies the requirement pyg_lib (from versions: none)
ERROR: No matching distribution found for pyg_lib


In [ ]:
import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, BatchNorm1d, ReLU
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_add_pool
import numpy as np
import os
import time
import psutil
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score

# === CONFIGURATION ===
DATASET_NAME = 'ENZYMES'  # Options: 'MUTAG', 'ENZYMES', 'IMDB-MULTI'
BATCH_SIZE = 64
HIDDEN_DIM = 32
DROPOUT = 0.15
EPOCHS = 1000  # lower for testing in Kaggle

# === CUSTOM TU DATASET LOADER ===
def get_dataset_from_files(name, root="/kaggle/input/datasets"):
    """
    Load TU dataset from local Kaggle files and build PyG Data objects
    """
    base_path = os.path.join(root, name)
    
    # Mandatory files
    A_path = os.path.join(base_path, f"{name}_A.txt")
    graph_labels_path = os.path.join(base_path, f"{name}_graph_labels.txt")
    graph_indicator_path = os.path.join(base_path, f"{name}_graph_indicator.txt")
    
    # Optional files
    node_labels_path = os.path.join(base_path, f"{name}_node_labels.txt")
    
    # Load edges (1-indexed in TU dataset)
    edges = np.genfromtxt(A_path, dtype=int) - 1
    # Convert to shape [num_edges, 2] if only one edge
    edges = edges.reshape(-1, 2) if edges.ndim == 1 else edges
    
    # Load graph indicators
    graph_indicator = np.loadtxt(graph_indicator_path, dtype=int) - 1
    num_graphs = graph_indicator.max() + 1
    
    # Load graph labels
    graph_labels = np.loadtxt(graph_labels_path, dtype=int)
    if graph_labels.min() == -1:
        graph_labels += 1  # shift to 0-based
    
    # Node features
    node_attributes_path = os.path.join(base_path, f"{name}_node_attributes.txt")
    node_labels_path = os.path.join(base_path, f"{name}_node_labels.txt")
    
    if os.path.exists(node_attributes_path):
        x_all = np.loadtxt(node_attributes_path, delimiter=',', dtype=float)
        x_all = torch.tensor(x_all, dtype=torch.float)
        num_node_features = x_all.shape[1]
    elif os.path.exists(node_labels_path):
        node_labels = np.loadtxt(node_labels_path, dtype=int)
        node_labels = torch.tensor(node_labels, dtype=torch.long).unsqueeze(1)
        num_node_features = int(node_labels.max()) + 1
        x_all = torch.nn.functional.one_hot(node_labels.squeeze(), num_classes=num_node_features).float()
    else:
        # fallback: one-hot degree vector
        degrees = np.zeros(graph_indicator.shape[0], dtype=int)
        for e in edges:
            degrees[e[0]] += 1
            degrees[e[1]] += 1
        num_node_features = int(degrees.max()) + 1
        x_all = torch.nn.functional.one_hot(torch.tensor(degrees), num_classes=num_node_features).float()

    # Build Data objects per graph
    dataset = []
    for g in range(num_graphs):
        node_mask = graph_indicator == g
        node_idx = np.where(node_mask)[0]
        idx_map = {old: new for new, old in enumerate(node_idx)}
        
        # Select edges for this graph
        mask_edges = [e for e in edges if e[0] in idx_map and e[1] in idx_map]
        if len(mask_edges) == 0:
            edge_index = torch.empty((2,0), dtype=torch.long)
        else:
            edge_index = torch.tensor([[idx_map[e[0]], idx_map[e[1]]] for e in mask_edges], dtype=torch.long).T
        
        x_graph = x_all[node_mask]
        y_graph = torch.tensor([graph_labels[g]], dtype=torch.long)
        
        dataset.append(Data(x=x_graph, edge_index=edge_index, y=y_graph))
    
    # Shuffle and split 80/20
    np.random.seed(12345)
    np.random.shuffle(dataset)
    train_size = int(0.8 * len(dataset))
    train_dataset = dataset[:train_size]
    test_dataset = dataset[train_size:]
    
    return dataset, train_dataset, test_dataset

# === LOAD DATA ===
dataset, train_dataset, test_dataset = get_dataset_from_files(DATASET_NAME, root="/kaggle/input/graph-dataset")
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Dataset: {DATASET_NAME}")
print(f"Number of graphs: {len(dataset)}")
print(f"Number of classes: {max([d.y.item() for d in dataset]) + 1}")
print(f"Number of node features: {dataset[0].x.shape[1]}")
print(f"Embedding dimensions: {HIDDEN_DIM}")
print("="*40)

# === GIN MODEL ===
class GIN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.15, num_layers=3):
        super().__init__()
        self.dropout = dropout
        self.num_layers = num_layers
        
        self.convs = torch.nn.ModuleList()
        for i in range(num_layers):
            input_dim = in_channels if i == 0 else hidden_channels
            mlp = Sequential(
                Linear(input_dim, hidden_channels),
                BatchNorm1d(hidden_channels),
                ReLU(),
                Linear(hidden_channels, hidden_channels),
                ReLU()
            )
            conv = GINConv(mlp)
            self.convs.append(conv)
        
        # Classifier
        self.lin1 = Linear(hidden_channels*num_layers, hidden_channels*num_layers)
        self.lin2 = Linear(hidden_channels*num_layers, out_channels)
    
    def forward(self, x, edge_index, batch):
        layer_embeddings = []
        for conv in self.convs:
            x = conv(x, edge_index)
            x = x.relu()
            x = F.dropout(x, p=self.dropout, training=self.training)
            layer_embeddings.append(x)
        
        pooled = [global_add_pool(h, batch) for h in layer_embeddings]
        graph_embedding = torch.cat(pooled, dim=1)
        
        h = self.lin1(graph_embedding)
        h = h.relu()
        h = F.dropout(h, p=0.5, training=self.training)
        out = self.lin2(h)
        
        return F.log_softmax(out, dim=1), F.softmax(out, dim=1), graph_embedding

# === SETUP MODEL ===
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GIN(dataset[0].x.shape[1], HIDDEN_DIM, max([d.y.item() for d in dataset]) + 1, DROPOUT).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# === TRAIN & TEST FUNCTIONS ===
def train():
    model.train()
    total_loss = 0
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        out, _, _ = model(data.x, data.edge_index, data.batch)
        loss = F.nll_loss(out, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

def test(loader):
    model.eval()
    all_preds, all_labels, all_probs, embeddings_list = [], [], [], []
    for data in loader:
        data = data.to(device)
        out, probs, embeddings = model(data.x, data.edge_index, data.batch)
        pred = probs.argmax(dim=1)
        all_preds.append(pred.detach().cpu())
        all_labels.append(data.y.detach().cpu())
        all_probs.append(probs.detach().cpu())
        embeddings_list.append(embeddings.detach().cpu())
    y_pred = torch.cat(all_preds).numpy()
    y_true = torch.cat(all_labels).numpy()
    y_probs = torch.cat(all_probs).numpy()
    final_embeddings = torch.cat(embeddings_list)
    return y_true, y_pred, y_probs, final_embeddings

# === TRAINING LOOP ===
start_train = time.time()
for epoch in range(1, EPOCHS+1):
    loss = train()
    if epoch % 10 == 0:
        y_true_monitor, y_pred_monitor, _, _ = test(test_loader)
        acc_monitor = accuracy_score(y_true_monitor, y_pred_monitor)
        print(f"Epoch {epoch:03d} | Loss {loss:.4f} | Test Acc {acc_monitor:.4f}")
end_train = time.time()

# === FINAL EVALUATION ===
y_true, y_pred, y_probs, final_embeddings = test(test_loader)
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='weighted')
try:
    auc = roc_auc_score(y_true, y_probs, multi_class='ovr')
except:
    auc = 0.0
memory_used = psutil.Process(os.getpid()).memory_info().rss / 1024**2

print("\n" + "="*40)
print("FINAL RESULTS")
print("="*40)
print(f"Dataset: {DATASET_NAME}")
print(f"Accuracy: {acc:.4f}")
print(f"F1-score: {f1:.4f}")
print(f"AUC: {auc:.4f}")
print(f"Training Time: {end_train-start_train:.2f} sec")
print(f"Memory Used: {memory_used:.2f} MB")
print("="*40)

Dataset: ENZYMES
Number of graphs: 600
Number of classes: 7
Number of node features: 18
Embedding dimensions: 32
Epoch 010 | Loss 1.7165 | Test Acc 0.2417
Epoch 020 | Loss 1.5613 | Test Acc 0.3333
Epoch 030 | Loss 1.5030 | Test Acc 0.3417
Epoch 040 | Loss 1.4307 | Test Acc 0.4417
Epoch 050 | Loss 1.3768 | Test Acc 0.2583
Epoch 060 | Loss 1.3230 | Test Acc 0.3750
Epoch 070 | Loss 1.2513 | Test Acc 0.4333
Epoch 080 | Loss 1.2500 | Test Acc 0.3417
Epoch 090 | Loss 1.0952 | Test Acc 0.4917
Epoch 100 | Loss 1.1032 | Test Acc 0.3417
Epoch 110 | Loss 1.0500 | Test Acc 0.4250
Epoch 120 | Loss 1.0539 | Test Acc 0.4917
Epoch 130 | Loss 1.0671 | Test Acc 0.4667
Epoch 140 | Loss 0.9081 | Test Acc 0.5333
Epoch 150 | Loss 1.0015 | Test Acc 0.5083
Epoch 160 | Loss 0.9673 | Test Acc 0.4833
Epoch 170 | Loss 0.9045 | Test Acc 0.5000
Epoch 180 | Loss 0.8646 | Test Acc 0.5750
Epoch 190 | Loss 0.8452 | Test Acc 0.5250
Epoch 200 | Loss 0.9335 | Test Acc 0.4917
Epoch 210 | Loss 0.9627 | Test Acc 0.4250
Epoch

there's something wrong with the computation of auc ^^

In [ ]:
import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, BatchNorm1d, ReLU
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_add_pool
import numpy as np
import os
import time
import psutil
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score

# === CONFIGURATION ===
DATASET_NAME = 'MUTAG'  # Options: 'MUTAG', 'ENZYMES', 'IMDB-MULTI'
BATCH_SIZE = 64
HIDDEN_DIM = 32
DROPOUT = 0.15
EPOCHS = 1000  # lower for testing in Kaggle

# === CUSTOM TU DATASET LOADER ===
def get_dataset_from_files(name, root="/kaggle/input/datasets"):
    """
    Load TU dataset from local Kaggle files and build PyG Data objects
    """
    base_path = os.path.join(root, name)
    
    # Mandatory files
    A_path = os.path.join(base_path, f"{name}_A.txt")
    graph_labels_path = os.path.join(base_path, f"{name}_graph_labels.txt")
    graph_indicator_path = os.path.join(base_path, f"{name}_graph_indicator.txt")
    
    # Optional files
    node_labels_path = os.path.join(base_path, f"{name}_node_labels.txt")
    
    # Load edges (1-indexed in TU dataset)
    edges = np.genfromtxt(A_path, dtype=int) - 1
    # Convert to shape [num_edges, 2] if only one edge
    edges = edges.reshape(-1, 2) if edges.ndim == 1 else edges
    
    # Load graph indicators
    graph_indicator = np.loadtxt(graph_indicator_path, dtype=int) - 1
    num_graphs = graph_indicator.max() + 1
    
    # Load graph labels
    graph_labels = np.loadtxt(graph_labels_path, dtype=int)
    if graph_labels.min() == -1:
        graph_labels += 1  # shift to 0-based
    
    # Node features
    node_attributes_path = os.path.join(base_path, f"{name}_node_attributes.txt")
    node_labels_path = os.path.join(base_path, f"{name}_node_labels.txt")
    
    if os.path.exists(node_attributes_path):
        x_all = np.loadtxt(node_attributes_path, delimiter=',', dtype=float)
        x_all = torch.tensor(x_all, dtype=torch.float)
        num_node_features = x_all.shape[1]
    elif os.path.exists(node_labels_path):
        node_labels = np.loadtxt(node_labels_path, dtype=int)
        node_labels = torch.tensor(node_labels, dtype=torch.long).unsqueeze(1)
        num_node_features = int(node_labels.max()) + 1
        x_all = torch.nn.functional.one_hot(node_labels.squeeze(), num_classes=num_node_features).float()
    else:
        # fallback: one-hot degree vector
        degrees = np.zeros(graph_indicator.shape[0], dtype=int)
        for e in edges:
            degrees[e[0]] += 1
            degrees[e[1]] += 1
        num_node_features = int(degrees.max()) + 1
        x_all = torch.nn.functional.one_hot(torch.tensor(degrees), num_classes=num_node_features).float()

    # Build Data objects per graph
    dataset = []
    for g in range(num_graphs):
        node_mask = graph_indicator == g
        node_idx = np.where(node_mask)[0]
        idx_map = {old: new for new, old in enumerate(node_idx)}
        
        # Select edges for this graph
        mask_edges = [e for e in edges if e[0] in idx_map and e[1] in idx_map]
        if len(mask_edges) == 0:
            edge_index = torch.empty((2,0), dtype=torch.long)
        else:
            edge_index = torch.tensor([[idx_map[e[0]], idx_map[e[1]]] for e in mask_edges], dtype=torch.long).T
        
        x_graph = x_all[node_mask]
        y_graph = torch.tensor([graph_labels[g]], dtype=torch.long)
        
        dataset.append(Data(x=x_graph, edge_index=edge_index, y=y_graph))
    
    # Shuffle and split 80/20
    np.random.seed(12345)
    np.random.shuffle(dataset)
    train_size = int(0.8 * len(dataset))
    train_dataset = dataset[:train_size]
    test_dataset = dataset[train_size:]
    
    return dataset, train_dataset, test_dataset

# === LOAD DATA ===
dataset, train_dataset, test_dataset = get_dataset_from_files(DATASET_NAME, root="/kaggle/input/graph-dataset")
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Dataset: {DATASET_NAME}")
print(f"Number of graphs: {len(dataset)}")
print(f"Number of classes: {max([d.y.item() for d in dataset]) + 1}")
print(f"Number of node features: {dataset[0].x.shape[1]}")
print(f"Embedding dimensions: {HIDDEN_DIM}")
print("="*40)

# === GIN MODEL ===
class GIN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.15, num_layers=3):
        super().__init__()
        self.dropout = dropout
        self.num_layers = num_layers
        
        self.convs = torch.nn.ModuleList()
        for i in range(num_layers):
            input_dim = in_channels if i == 0 else hidden_channels
            mlp = Sequential(
                Linear(input_dim, hidden_channels),
                BatchNorm1d(hidden_channels),
                ReLU(),
                Linear(hidden_channels, hidden_channels),
                ReLU()
            )
            conv = GINConv(mlp)
            self.convs.append(conv)
        
        # Classifier
        self.lin1 = Linear(hidden_channels*num_layers, hidden_channels*num_layers)
        self.lin2 = Linear(hidden_channels*num_layers, out_channels)
    
    def forward(self, x, edge_index, batch):
        layer_embeddings = []
        for conv in self.convs:
            x = conv(x, edge_index)
            x = x.relu()
            x = F.dropout(x, p=self.dropout, training=self.training)
            layer_embeddings.append(x)
        
        pooled = [global_add_pool(h, batch) for h in layer_embeddings]
        graph_embedding = torch.cat(pooled, dim=1)
        
        h = self.lin1(graph_embedding)
        h = h.relu()
        h = F.dropout(h, p=0.5, training=self.training)
        out = self.lin2(h)
        
        return F.log_softmax(out, dim=1), F.softmax(out, dim=1), graph_embedding

# === SETUP MODEL ===
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GIN(dataset[0].x.shape[1], HIDDEN_DIM, max([d.y.item() for d in dataset]) + 1, DROPOUT).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# === TRAIN & TEST FUNCTIONS ===
def train():
    model.train()
    total_loss = 0
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        out, _, _ = model(data.x, data.edge_index, data.batch)
        loss = F.nll_loss(out, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

def test(loader):
    model.eval()
    all_preds, all_labels, all_probs, embeddings_list = [], [], [], []
    for data in loader:
        data = data.to(device)
        out, probs, embeddings = model(data.x, data.edge_index, data.batch)
        pred = probs.argmax(dim=1)
        all_preds.append(pred.detach().cpu())
        all_labels.append(data.y.detach().cpu())
        all_probs.append(probs.detach().cpu())
        embeddings_list.append(embeddings.detach().cpu())
    y_pred = torch.cat(all_preds).numpy()
    y_true = torch.cat(all_labels).numpy()
    y_probs = torch.cat(all_probs).numpy()
    final_embeddings = torch.cat(embeddings_list)
    return y_true, y_pred, y_probs, final_embeddings

# === TRAINING LOOP ===
start_train = time.time()
for epoch in range(1, EPOCHS+1):
    loss = train()
    if epoch % 10 == 0:
        y_true_monitor, y_pred_monitor, _, _ = test(test_loader)
        acc_monitor = accuracy_score(y_true_monitor, y_pred_monitor)
        print(f"Epoch {epoch:03d} | Loss {loss:.4f} | Test Acc {acc_monitor:.4f}")
end_train = time.time()

# === FINAL EVALUATION ===
y_true, y_pred, y_probs, final_embeddings = test(test_loader)
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='weighted')
try:
    auc = roc_auc_score(y_true, y_probs, multi_class='ovr')
except:
    auc = 0.0
memory_used = psutil.Process(os.getpid()).memory_info().rss / 1024**2

print("\n" + "="*40)
print("FINAL RESULTS")
print("="*40)
print(f"Dataset: {DATASET_NAME}")
print(f"Accuracy: {acc:.4f}")
print(f"F1-score: {f1:.4f}")
print(f"AUC: {auc:.4f}")
print(f"Training Time: {end_train-start_train:.2f} sec")
print(f"Memory Used: {memory_used:.2f} MB")
print("="*40)

Dataset: MUTAG
Number of graphs: 188
Number of classes: 3
Number of node features: 7
Embedding dimensions: 32
Epoch 010 | Loss 0.5238 | Test Acc 0.5526
Epoch 020 | Loss 0.5018 | Test Acc 0.6316
Epoch 030 | Loss 0.4673 | Test Acc 0.6842
Epoch 040 | Loss 0.4609 | Test Acc 0.7105
Epoch 050 | Loss 0.5177 | Test Acc 0.6842
Epoch 060 | Loss 0.4520 | Test Acc 0.7105
Epoch 070 | Loss 0.4406 | Test Acc 0.6842
Epoch 080 | Loss 0.4408 | Test Acc 0.6842
Epoch 090 | Loss 0.3991 | Test Acc 0.6842
Epoch 100 | Loss 0.4526 | Test Acc 0.7632
Epoch 110 | Loss 0.4589 | Test Acc 0.6842
Epoch 120 | Loss 0.4033 | Test Acc 0.6842
Epoch 130 | Loss 0.5843 | Test Acc 0.7895
Epoch 140 | Loss 0.4741 | Test Acc 0.8158
Epoch 150 | Loss 0.3751 | Test Acc 0.8158
Epoch 160 | Loss 0.3905 | Test Acc 0.8158
Epoch 170 | Loss 0.4624 | Test Acc 0.8158
Epoch 180 | Loss 0.3922 | Test Acc 0.8684
Epoch 190 | Loss 0.3625 | Test Acc 0.8158
Epoch 200 | Loss 0.4287 | Test Acc 0.8158
Epoch 210 | Loss 0.4067 | Test Acc 0.8158
Epoch 22

In [ ]:
import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, BatchNorm1d, ReLU
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_add_pool
import numpy as np
import os
import time
import psutil
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score

# === CONFIGURATION ===
DATASET_NAME = 'IMDB-MULTI'  # Options: 'MUTAG', 'ENZYMES', 'IMDB-MULTI'
BATCH_SIZE = 64
HIDDEN_DIM = 32
DROPOUT = 0.15
EPOCHS = 1000  # lower for testing in Kaggle

# === CUSTOM TU DATASET LOADER ===
def get_dataset_from_files(name, root="/kaggle/input/datasets"):
    """
    Load TU dataset from local Kaggle files and build PyG Data objects
    """
    base_path = os.path.join(root, name)
    
    # Mandatory files
    A_path = os.path.join(base_path, f"{name}_A.txt")
    graph_labels_path = os.path.join(base_path, f"{name}_graph_labels.txt")
    graph_indicator_path = os.path.join(base_path, f"{name}_graph_indicator.txt")
    
    # Optional files
    node_labels_path = os.path.join(base_path, f"{name}_node_labels.txt")
    
    # Load edges (1-indexed in TU dataset)
    edges = np.genfromtxt(A_path, dtype=int) - 1
    # Convert to shape [num_edges, 2] if only one edge
    edges = edges.reshape(-1, 2) if edges.ndim == 1 else edges
    
    # Load graph indicators
    graph_indicator = np.loadtxt(graph_indicator_path, dtype=int) - 1
    num_graphs = graph_indicator.max() + 1
    
    # Load graph labels
    graph_labels = np.loadtxt(graph_labels_path, dtype=int)
    if graph_labels.min() == -1:
        graph_labels += 1  # shift to 0-based
    
    # Node features
    node_attributes_path = os.path.join(base_path, f"{name}_node_attributes.txt")
    node_labels_path = os.path.join(base_path, f"{name}_node_labels.txt")
    
    if os.path.exists(node_attributes_path):
        x_all = np.loadtxt(node_attributes_path, delimiter=',', dtype=float)
        x_all = torch.tensor(x_all, dtype=torch.float)
        num_node_features = x_all.shape[1]
    elif os.path.exists(node_labels_path):
        node_labels = np.loadtxt(node_labels_path, dtype=int)
        node_labels = torch.tensor(node_labels, dtype=torch.long).unsqueeze(1)
        num_node_features = int(node_labels.max()) + 1
        x_all = torch.nn.functional.one_hot(node_labels.squeeze(), num_classes=num_node_features).float()
    else:
        # fallback for IMDB-MULTI: just use degree as scalar
        num_nodes = graph_indicator.shape[0]
        degrees = np.zeros(num_nodes, dtype=int)
        for e in edges:
            degrees[e[0]] += 1
            degrees[e[1]] += 1
        x_all = torch.tensor(degrees, dtype=torch.float).unsqueeze(1)  # <--- memory-safe

    # Build Data objects per graph
    dataset = []
    for g in range(num_graphs):
        node_mask = graph_indicator == g
        node_idx = np.where(node_mask)[0]
        idx_map = {old: new for new, old in enumerate(node_idx)}
        
        # Select edges for this graph
        mask_edges = [e for e in edges if e[0] in idx_map and e[1] in idx_map]
        if len(mask_edges) == 0:
            edge_index = torch.empty((2,0), dtype=torch.long)
        else:
            edge_index = torch.tensor([[idx_map[e[0]], idx_map[e[1]]] for e in mask_edges], dtype=torch.long).T
        
        x_graph = x_all[node_mask]
        y_graph = torch.tensor([graph_labels[g]], dtype=torch.long)
        
        dataset.append(Data(x=x_graph, edge_index=edge_index, y=y_graph))
    
    # Shuffle and split 80/20
    np.random.seed(12345)
    np.random.shuffle(dataset)
    train_size = int(0.8 * len(dataset))
    train_dataset = dataset[:train_size]
    test_dataset = dataset[train_size:]
    
    return dataset, train_dataset, test_dataset

# === LOAD DATA ===
dataset, train_dataset, test_dataset = get_dataset_from_files(DATASET_NAME, root="/kaggle/input/graph-dataset")
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Dataset: {DATASET_NAME}")
print(f"Number of graphs: {len(dataset)}")
print(f"Number of classes: {max([d.y.item() for d in dataset]) + 1}")
print(f"Number of node features: {dataset[0].x.shape[1]}")
print(f"Embedding dimensions: {HIDDEN_DIM}")
print("="*40)

# === GIN MODEL ===
class GIN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.15, num_layers=3):
        super().__init__()
        self.dropout = dropout
        self.num_layers = num_layers
        
        self.convs = torch.nn.ModuleList()
        for i in range(num_layers):
            input_dim = in_channels if i == 0 else hidden_channels
            mlp = Sequential(
                Linear(input_dim, hidden_channels),
                BatchNorm1d(hidden_channels),
                ReLU(),
                Linear(hidden_channels, hidden_channels),
                ReLU()
            )
            conv = GINConv(mlp)
            self.convs.append(conv)
        
        # Classifier
        self.lin1 = Linear(hidden_channels*num_layers, hidden_channels*num_layers)
        self.lin2 = Linear(hidden_channels*num_layers, out_channels)
    
    def forward(self, x, edge_index, batch):
        layer_embeddings = []
        for conv in self.convs:
            x = conv(x, edge_index)
            x = x.relu()
            x = F.dropout(x, p=self.dropout, training=self.training)
            layer_embeddings.append(x)
        
        pooled = [global_add_pool(h, batch) for h in layer_embeddings]
        graph_embedding = torch.cat(pooled, dim=1)
        
        h = self.lin1(graph_embedding)
        h = h.relu()
        h = F.dropout(h, p=0.5, training=self.training)
        out = self.lin2(h)
        
        return F.log_softmax(out, dim=1), F.softmax(out, dim=1), graph_embedding

# === SETUP MODEL ===
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GIN(dataset[0].x.shape[1], HIDDEN_DIM, max([d.y.item() for d in dataset]) + 1, DROPOUT).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# === TRAIN & TEST FUNCTIONS ===
def train():
    model.train()
    total_loss = 0
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        out, _, _ = model(data.x, data.edge_index, data.batch)
        loss = F.nll_loss(out, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

def test(loader):
    model.eval()
    all_preds, all_labels, all_probs, embeddings_list = [], [], [], []
    for data in loader:
        data = data.to(device)
        out, probs, embeddings = model(data.x, data.edge_index, data.batch)
        pred = probs.argmax(dim=1)
        all_preds.append(pred.detach().cpu())
        all_labels.append(data.y.detach().cpu())
        all_probs.append(probs.detach().cpu())
        embeddings_list.append(embeddings.detach().cpu())
    y_pred = torch.cat(all_preds).numpy()
    y_true = torch.cat(all_labels).numpy()
    y_probs = torch.cat(all_probs).numpy()
    final_embeddings = torch.cat(embeddings_list)
    return y_true, y_pred, y_probs, final_embeddings

# === TRAINING LOOP ===
start_train = time.time()
for epoch in range(1, EPOCHS+1):
    loss = train()
    if epoch % 10 == 0:
        y_true_monitor, y_pred_monitor, _, _ = test(test_loader)
        acc_monitor = accuracy_score(y_true_monitor, y_pred_monitor)
        print(f"Epoch {epoch:03d} | Loss {loss:.4f} | Test Acc {acc_monitor:.4f}")
end_train = time.time()

# === FINAL EVALUATION ===
y_true, y_pred, y_probs, final_embeddings = test(test_loader)
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='weighted')
try:
    auc = roc_auc_score(y_true, y_probs, multi_class='ovr')
except:
    auc = 0.0
memory_used = psutil.Process(os.getpid()).memory_info().rss / 1024**2

print("\n" + "="*40)
print("FINAL RESULTS")
print("="*40)
print(f"Dataset: {DATASET_NAME}")
print(f"Accuracy: {acc:.4f}")
print(f"F1-score: {f1:.4f}")
print(f"AUC: {auc:.4f}")
print(f"Training Time: {end_train-start_train:.2f} sec")
print(f"Memory Used: {memory_used:.2f} MB")
print("="*40)

Dataset: IMDB-MULTI
Number of graphs: 1500
Number of classes: 4
Number of node features: 1
Embedding dimensions: 32
Epoch 010 | Loss 1.0637 | Test Acc 0.3567
Epoch 020 | Loss 1.0627 | Test Acc 0.3500
Epoch 030 | Loss 1.0798 | Test Acc 0.3500
Epoch 040 | Loss 1.0641 | Test Acc 0.3500
Epoch 050 | Loss 1.0602 | Test Acc 0.3500
Epoch 060 | Loss 1.0724 | Test Acc 0.3500
Epoch 070 | Loss 1.0599 | Test Acc 0.2933
Epoch 080 | Loss 1.0510 | Test Acc 0.3500
Epoch 090 | Loss 1.0610 | Test Acc 0.3533
Epoch 100 | Loss 1.0594 | Test Acc 0.3500
Epoch 110 | Loss 1.0633 | Test Acc 0.3133
Epoch 120 | Loss 1.0643 | Test Acc 0.3567
Epoch 130 | Loss 1.0562 | Test Acc 0.3500
Epoch 140 | Loss 1.0601 | Test Acc 0.3500
Epoch 150 | Loss 1.0557 | Test Acc 0.3500
Epoch 160 | Loss 1.0617 | Test Acc 0.3500
Epoch 170 | Loss 1.0630 | Test Acc 0.3467
Epoch 180 | Loss 1.0512 | Test Acc 0.3500
Epoch 190 | Loss 1.0608 | Test Acc 0.3500
Epoch 200 | Loss 1.0462 | Test Acc 0.3500
Epoch 210 | Loss 1.0600 | Test Acc 0.3433
Ep

**GIN expects informative node features. Without them, it underperforms. IMDB-MULTI has no features, so degree alone isn’t enough. One-hot degree is memory-intensive; scalar degree is memory-safe but weak.**

In [2]:
#=== ENZYMES 5-LAYER GIN =====
import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, BatchNorm1d, ReLU
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_add_pool
import numpy as np
import os
import time
import psutil
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score

# === CONFIGURATION ===
DATASET_NAME = 'ENZYMES'  # Options: 'MUTAG', 'ENZYMES', 'IMDB-MULTI'
BATCH_SIZE = 64
HIDDEN_DIM = 32
DROPOUT = 0.15
EPOCHS = 1000  # lower for testing in Kaggle

# === CUSTOM TU DATASET LOADER ===
def get_dataset_from_files(name, root="/kaggle/input/datasets"):
    """
    Load TU dataset from local Kaggle files and build PyG Data objects
    """
    base_path = os.path.join(root, name)
    
    # Mandatory files
    A_path = os.path.join(base_path, f"{name}_A.txt")
    graph_labels_path = os.path.join(base_path, f"{name}_graph_labels.txt")
    graph_indicator_path = os.path.join(base_path, f"{name}_graph_indicator.txt")
    
    # Optional files
    node_labels_path = os.path.join(base_path, f"{name}_node_labels.txt")
    
    # Load edges (1-indexed in TU dataset)
    edges = np.genfromtxt(A_path, dtype=int) - 1
    # Convert to shape [num_edges, 2] if only one edge
    edges = edges.reshape(-1, 2) if edges.ndim == 1 else edges
    
    # Load graph indicators
    graph_indicator = np.loadtxt(graph_indicator_path, dtype=int) - 1
    num_graphs = graph_indicator.max() + 1
    
    # Load graph labels
    graph_labels = np.loadtxt(graph_labels_path, dtype=int)
    if graph_labels.min() == -1:
        graph_labels += 1  # shift to 0-based
    
    # Node features
    node_attributes_path = os.path.join(base_path, f"{name}_node_attributes.txt")
    node_labels_path = os.path.join(base_path, f"{name}_node_labels.txt")
    
    if os.path.exists(node_attributes_path):
        x_all = np.loadtxt(node_attributes_path, delimiter=',', dtype=float)
        x_all = torch.tensor(x_all, dtype=torch.float)
        num_node_features = x_all.shape[1]
    elif os.path.exists(node_labels_path):
        node_labels = np.loadtxt(node_labels_path, dtype=int)
        node_labels = torch.tensor(node_labels, dtype=torch.long).unsqueeze(1)
        num_node_features = int(node_labels.max()) + 1
        x_all = torch.nn.functional.one_hot(node_labels.squeeze(), num_classes=num_node_features).float()
    else:
        # fallback: one-hot degree vector
        degrees = np.zeros(graph_indicator.shape[0], dtype=int)
        for e in edges:
            degrees[e[0]] += 1
            degrees[e[1]] += 1
        num_node_features = int(degrees.max()) + 1
        x_all = torch.nn.functional.one_hot(torch.tensor(degrees), num_classes=num_node_features).float()

    # Build Data objects per graph
    dataset = []
    for g in range(num_graphs):
        node_mask = graph_indicator == g
        node_idx = np.where(node_mask)[0]
        idx_map = {old: new for new, old in enumerate(node_idx)}
        
        # Select edges for this graph
        mask_edges = [e for e in edges if e[0] in idx_map and e[1] in idx_map]
        if len(mask_edges) == 0:
            edge_index = torch.empty((2,0), dtype=torch.long)
        else:
            edge_index = torch.tensor([[idx_map[e[0]], idx_map[e[1]]] for e in mask_edges], dtype=torch.long).T
        
        x_graph = x_all[node_mask]
        y_graph = torch.tensor([graph_labels[g]], dtype=torch.long)
        
        dataset.append(Data(x=x_graph, edge_index=edge_index, y=y_graph))
    
    # Shuffle and split 80/20
    np.random.seed(12345)
    np.random.shuffle(dataset)
    train_size = int(0.8 * len(dataset))
    train_dataset = dataset[:train_size]
    test_dataset = dataset[train_size:]
    
    return dataset, train_dataset, test_dataset

# === LOAD DATA ===
dataset, train_dataset, test_dataset = get_dataset_from_files(DATASET_NAME, root="/kaggle/input/graph-dataset")
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Dataset: {DATASET_NAME}")
print(f"Number of graphs: {len(dataset)}")
print(f"Number of classes: {max([d.y.item() for d in dataset]) + 1}")
print(f"Number of node features: {dataset[0].x.shape[1]}")
print(f"Embedding dimensions: {HIDDEN_DIM}")
print("="*40)

# === GIN MODEL ===
class GIN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.15, num_layers=5):
        super().__init__()
        self.dropout = dropout
        self.num_layers = num_layers
        
        self.convs = torch.nn.ModuleList()
        for i in range(num_layers):
            input_dim = in_channels if i == 0 else hidden_channels
            mlp = Sequential(
                Linear(input_dim, hidden_channels),
                BatchNorm1d(hidden_channels),
                ReLU(),
                Linear(hidden_channels, hidden_channels),
                ReLU()
            )
            conv = GINConv(mlp)
            self.convs.append(conv)
        
        # Classifier
        self.lin1 = Linear(hidden_channels*num_layers, hidden_channels*num_layers)
        self.lin2 = Linear(hidden_channels*num_layers, out_channels)
    
    def forward(self, x, edge_index, batch):
        layer_embeddings = []
        for conv in self.convs:
            x = conv(x, edge_index)
            x = x.relu()
            x = F.dropout(x, p=self.dropout, training=self.training)
            layer_embeddings.append(x)
        
        pooled = [global_add_pool(h, batch) for h in layer_embeddings]
        graph_embedding = torch.cat(pooled, dim=1)
        
        h = self.lin1(graph_embedding)
        h = h.relu()
        h = F.dropout(h, p=0.5, training=self.training)
        out = self.lin2(h)
        
        return F.log_softmax(out, dim=1), F.softmax(out, dim=1), graph_embedding

# === SETUP MODEL ===
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GIN(dataset[0].x.shape[1], HIDDEN_DIM, max([d.y.item() for d in dataset]) + 1, DROPOUT).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# === TRAIN & TEST FUNCTIONS ===
def train():
    model.train()
    total_loss = 0
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        out, _, _ = model(data.x, data.edge_index, data.batch)
        loss = F.nll_loss(out, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

def test(loader):
    model.eval()
    all_preds, all_labels, all_probs, embeddings_list = [], [], [], []
    for data in loader:
        data = data.to(device)
        out, probs, embeddings = model(data.x, data.edge_index, data.batch)
        pred = probs.argmax(dim=1)
        all_preds.append(pred.detach().cpu())
        all_labels.append(data.y.detach().cpu())
        all_probs.append(probs.detach().cpu())
        embeddings_list.append(embeddings.detach().cpu())
    y_pred = torch.cat(all_preds).numpy()
    y_true = torch.cat(all_labels).numpy()
    y_probs = torch.cat(all_probs).numpy()
    final_embeddings = torch.cat(embeddings_list)
    return y_true, y_pred, y_probs, final_embeddings

# === TRAINING LOOP ===
start_train = time.time()
for epoch in range(1, EPOCHS+1):
    loss = train()
    if epoch % 10 == 0:
        y_true_monitor, y_pred_monitor, _, _ = test(test_loader)
        acc_monitor = accuracy_score(y_true_monitor, y_pred_monitor)
        print(f"Epoch {epoch:03d} | Loss {loss:.4f} | Test Acc {acc_monitor:.4f}")
end_train = time.time()

# === FINAL EVALUATION ===
y_true, y_pred, y_probs, final_embeddings = test(test_loader)
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='weighted')
try:
    auc = roc_auc_score(y_true, y_probs, multi_class='ovr')
except:
    auc = 0.0
memory_used = psutil.Process(os.getpid()).memory_info().rss / 1024**2

print("\n" + "="*40)
print("FINAL RESULTS")
print("="*40)
print(f"Dataset: {DATASET_NAME}")
print(f"Accuracy: {acc:.4f}")
print(f"F1-score: {f1:.4f}")
print(f"AUC: {auc:.4f}")
print(f"Training Time: {end_train-start_train:.2f} sec")
print(f"Memory Used: {memory_used:.2f} MB")
print("="*40)

Dataset: ENZYMES
Number of graphs: 600
Number of classes: 7
Number of node features: 18
Embedding dimensions: 32
Epoch 010 | Loss 1.6837 | Test Acc 0.2417
Epoch 020 | Loss 1.6356 | Test Acc 0.2500
Epoch 030 | Loss 1.5346 | Test Acc 0.3417
Epoch 040 | Loss 1.5567 | Test Acc 0.2167
Epoch 050 | Loss 1.4519 | Test Acc 0.4000
Epoch 060 | Loss 1.4484 | Test Acc 0.4083
Epoch 070 | Loss 1.2126 | Test Acc 0.4417
Epoch 080 | Loss 1.3291 | Test Acc 0.4083
Epoch 090 | Loss 1.2092 | Test Acc 0.3500
Epoch 100 | Loss 1.2002 | Test Acc 0.3833
Epoch 110 | Loss 1.1600 | Test Acc 0.4000
Epoch 120 | Loss 1.0670 | Test Acc 0.5417
Epoch 130 | Loss 1.1171 | Test Acc 0.4583
Epoch 140 | Loss 1.0321 | Test Acc 0.5583
Epoch 150 | Loss 0.8695 | Test Acc 0.5500
Epoch 160 | Loss 0.9894 | Test Acc 0.5083
Epoch 170 | Loss 0.9406 | Test Acc 0.5667
Epoch 180 | Loss 0.9017 | Test Acc 0.5583
Epoch 190 | Loss 0.8469 | Test Acc 0.4917
Epoch 200 | Loss 0.9963 | Test Acc 0.4417
Epoch 210 | Loss 0.8729 | Test Acc 0.5833
Epoch

In [3]:
#=== MUTAG 5-LAYER GIN =====
import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, BatchNorm1d, ReLU
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_add_pool
import numpy as np
import os
import time
import psutil
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score

# === CONFIGURATION ===
DATASET_NAME = 'MUTAG'  # Options: 'MUTAG', 'ENZYMES', 'IMDB-MULTI'
BATCH_SIZE = 64
HIDDEN_DIM = 32
DROPOUT = 0.15
EPOCHS = 1000  # lower for testing in Kaggle

# === CUSTOM TU DATASET LOADER ===
def get_dataset_from_files(name, root="/kaggle/input/datasets"):
    """
    Load TU dataset from local Kaggle files and build PyG Data objects
    """
    base_path = os.path.join(root, name)
    
    # Mandatory files
    A_path = os.path.join(base_path, f"{name}_A.txt")
    graph_labels_path = os.path.join(base_path, f"{name}_graph_labels.txt")
    graph_indicator_path = os.path.join(base_path, f"{name}_graph_indicator.txt")
    
    # Optional files
    node_labels_path = os.path.join(base_path, f"{name}_node_labels.txt")
    
    # Load edges (1-indexed in TU dataset)
    edges = np.genfromtxt(A_path, dtype=int) - 1
    # Convert to shape [num_edges, 2] if only one edge
    edges = edges.reshape(-1, 2) if edges.ndim == 1 else edges
    
    # Load graph indicators
    graph_indicator = np.loadtxt(graph_indicator_path, dtype=int) - 1
    num_graphs = graph_indicator.max() + 1
    
    # Load graph labels
    graph_labels = np.loadtxt(graph_labels_path, dtype=int)
    if graph_labels.min() == -1:
        graph_labels += 1  # shift to 0-based
    
    # Node features
    node_attributes_path = os.path.join(base_path, f"{name}_node_attributes.txt")
    node_labels_path = os.path.join(base_path, f"{name}_node_labels.txt")
    
    if os.path.exists(node_attributes_path):
        x_all = np.loadtxt(node_attributes_path, delimiter=',', dtype=float)
        x_all = torch.tensor(x_all, dtype=torch.float)
        num_node_features = x_all.shape[1]
    elif os.path.exists(node_labels_path):
        node_labels = np.loadtxt(node_labels_path, dtype=int)
        node_labels = torch.tensor(node_labels, dtype=torch.long).unsqueeze(1)
        num_node_features = int(node_labels.max()) + 1
        x_all = torch.nn.functional.one_hot(node_labels.squeeze(), num_classes=num_node_features).float()
    else:
        # fallback: one-hot degree vector
        degrees = np.zeros(graph_indicator.shape[0], dtype=int)
        for e in edges:
            degrees[e[0]] += 1
            degrees[e[1]] += 1
        num_node_features = int(degrees.max()) + 1
        x_all = torch.nn.functional.one_hot(torch.tensor(degrees), num_classes=num_node_features).float()

    # Build Data objects per graph
    dataset = []
    for g in range(num_graphs):
        node_mask = graph_indicator == g
        node_idx = np.where(node_mask)[0]
        idx_map = {old: new for new, old in enumerate(node_idx)}
        
        # Select edges for this graph
        mask_edges = [e for e in edges if e[0] in idx_map and e[1] in idx_map]
        if len(mask_edges) == 0:
            edge_index = torch.empty((2,0), dtype=torch.long)
        else:
            edge_index = torch.tensor([[idx_map[e[0]], idx_map[e[1]]] for e in mask_edges], dtype=torch.long).T
        
        x_graph = x_all[node_mask]
        y_graph = torch.tensor([graph_labels[g]], dtype=torch.long)
        
        dataset.append(Data(x=x_graph, edge_index=edge_index, y=y_graph))
    
    # Shuffle and split 80/20
    np.random.seed(12345)
    np.random.shuffle(dataset)
    train_size = int(0.8 * len(dataset))
    train_dataset = dataset[:train_size]
    test_dataset = dataset[train_size:]
    
    return dataset, train_dataset, test_dataset

# === LOAD DATA ===
dataset, train_dataset, test_dataset = get_dataset_from_files(DATASET_NAME, root="/kaggle/input/graph-dataset")
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Dataset: {DATASET_NAME}")
print(f"Number of graphs: {len(dataset)}")
print(f"Number of classes: {max([d.y.item() for d in dataset]) + 1}")
print(f"Number of node features: {dataset[0].x.shape[1]}")
print(f"Embedding dimensions: {HIDDEN_DIM}")
print("="*40)

# === GIN MODEL ===
class GIN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.15, num_layers=5):
        super().__init__()
        self.dropout = dropout
        self.num_layers = num_layers
        
        self.convs = torch.nn.ModuleList()
        for i in range(num_layers):
            input_dim = in_channels if i == 0 else hidden_channels
            mlp = Sequential(
                Linear(input_dim, hidden_channels),
                BatchNorm1d(hidden_channels),
                ReLU(),
                Linear(hidden_channels, hidden_channels),
                ReLU()
            )
            conv = GINConv(mlp)
            self.convs.append(conv)
        
        # Classifier
        self.lin1 = Linear(hidden_channels*num_layers, hidden_channels*num_layers)
        self.lin2 = Linear(hidden_channels*num_layers, out_channels)
    
    def forward(self, x, edge_index, batch):
        layer_embeddings = []
        for conv in self.convs:
            x = conv(x, edge_index)
            x = x.relu()
            x = F.dropout(x, p=self.dropout, training=self.training)
            layer_embeddings.append(x)
        
        pooled = [global_add_pool(h, batch) for h in layer_embeddings]
        graph_embedding = torch.cat(pooled, dim=1)
        
        h = self.lin1(graph_embedding)
        h = h.relu()
        h = F.dropout(h, p=0.5, training=self.training)
        out = self.lin2(h)
        
        return F.log_softmax(out, dim=1), F.softmax(out, dim=1), graph_embedding

# === SETUP MODEL ===
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GIN(dataset[0].x.shape[1], HIDDEN_DIM, max([d.y.item() for d in dataset]) + 1, DROPOUT).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# === TRAIN & TEST FUNCTIONS ===
def train():
    model.train()
    total_loss = 0
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        out, _, _ = model(data.x, data.edge_index, data.batch)
        loss = F.nll_loss(out, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

def test(loader):
    model.eval()
    all_preds, all_labels, all_probs, embeddings_list = [], [], [], []
    for data in loader:
        data = data.to(device)
        out, probs, embeddings = model(data.x, data.edge_index, data.batch)
        pred = probs.argmax(dim=1)
        all_preds.append(pred.detach().cpu())
        all_labels.append(data.y.detach().cpu())
        all_probs.append(probs.detach().cpu())
        embeddings_list.append(embeddings.detach().cpu())
    y_pred = torch.cat(all_preds).numpy()
    y_true = torch.cat(all_labels).numpy()
    y_probs = torch.cat(all_probs).numpy()
    final_embeddings = torch.cat(embeddings_list)
    return y_true, y_pred, y_probs, final_embeddings

# === TRAINING LOOP ===
start_train = time.time()
for epoch in range(1, EPOCHS+1):
    loss = train()
    if epoch % 10 == 0:
        y_true_monitor, y_pred_monitor, _, _ = test(test_loader)
        acc_monitor = accuracy_score(y_true_monitor, y_pred_monitor)
        print(f"Epoch {epoch:03d} | Loss {loss:.4f} | Test Acc {acc_monitor:.4f}")
end_train = time.time()

# === FINAL EVALUATION ===
y_true, y_pred, y_probs, final_embeddings = test(test_loader)
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='weighted')
try:
    auc = roc_auc_score(y_true, y_probs, multi_class='ovr')
except:
    auc = 0.0
memory_used = psutil.Process(os.getpid()).memory_info().rss / 1024**2

print("\n" + "="*40)
print("FINAL RESULTS")
print("="*40)
print(f"Dataset: {DATASET_NAME}")
print(f"Accuracy: {acc:.4f}")
print(f"F1-score: {f1:.4f}")
print(f"AUC: {auc:.4f}")
print(f"Training Time: {end_train-start_train:.2f} sec")
print(f"Memory Used: {memory_used:.2f} MB")
print("="*40)

Dataset: MUTAG
Number of graphs: 188
Number of classes: 3
Number of node features: 7
Embedding dimensions: 32
Epoch 010 | Loss 0.5347 | Test Acc 0.5526
Epoch 020 | Loss 0.4597 | Test Acc 0.6053
Epoch 030 | Loss 0.4639 | Test Acc 0.6842
Epoch 040 | Loss 0.5133 | Test Acc 0.6842
Epoch 050 | Loss 0.4770 | Test Acc 0.6579
Epoch 060 | Loss 0.4647 | Test Acc 0.6842
Epoch 070 | Loss 0.4265 | Test Acc 0.6579
Epoch 080 | Loss 0.4365 | Test Acc 0.6842
Epoch 090 | Loss 0.4223 | Test Acc 0.6842
Epoch 100 | Loss 0.4408 | Test Acc 0.6579
Epoch 110 | Loss 0.4742 | Test Acc 0.6842
Epoch 120 | Loss 0.4201 | Test Acc 0.6842
Epoch 130 | Loss 0.4100 | Test Acc 0.7368
Epoch 140 | Loss 0.4343 | Test Acc 0.8158
Epoch 150 | Loss 0.4418 | Test Acc 0.7895
Epoch 160 | Loss 0.4718 | Test Acc 0.8158
Epoch 170 | Loss 0.3865 | Test Acc 0.7632
Epoch 180 | Loss 0.3969 | Test Acc 0.8158
Epoch 190 | Loss 0.3798 | Test Acc 0.8158
Epoch 200 | Loss 0.4599 | Test Acc 0.8158
Epoch 210 | Loss 0.4959 | Test Acc 0.8684
Epoch 22

In [ ]:
#=== IMDB-MULTI 5-LAYER GIN =====
import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, BatchNorm1d, ReLU
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_add_pool
import numpy as np
import os
import time
import psutil
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score

# === CONFIGURATION ===
DATASET_NAME = 'IMDB-MULTI'  # Options: 'MUTAG', 'ENZYMES', 'IMDB-MULTI'
BATCH_SIZE = 64
HIDDEN_DIM = 32
DROPOUT = 0.15
EPOCHS = 1000  # lower for testing in Kaggle

# === CUSTOM TU DATASET LOADER ===
def get_dataset_from_files(name, root="/kaggle/input/datasets"):
    """
    Load TU dataset from local Kaggle files and build PyG Data objects
    """
    base_path = os.path.join(root, name)
    
    # Mandatory files
    A_path = os.path.join(base_path, f"{name}_A.txt")
    graph_labels_path = os.path.join(base_path, f"{name}_graph_labels.txt")
    graph_indicator_path = os.path.join(base_path, f"{name}_graph_indicator.txt")
    
    # Optional files
    node_labels_path = os.path.join(base_path, f"{name}_node_labels.txt")
    
    # Load edges (1-indexed in TU dataset)
    edges = np.genfromtxt(A_path, dtype=int) - 1
    # Convert to shape [num_edges, 2] if only one edge
    edges = edges.reshape(-1, 2) if edges.ndim == 1 else edges
    
    # Load graph indicators
    graph_indicator = np.loadtxt(graph_indicator_path, dtype=int) - 1
    num_graphs = graph_indicator.max() + 1
    
    # Load graph labels
    graph_labels = np.loadtxt(graph_labels_path, dtype=int)
    if graph_labels.min() == -1:
        graph_labels += 1  # shift to 0-based
    
    # Node features
    node_attributes_path = os.path.join(base_path, f"{name}_node_attributes.txt")
    node_labels_path = os.path.join(base_path, f"{name}_node_labels.txt")
    
    if os.path.exists(node_attributes_path):
        x_all = np.loadtxt(node_attributes_path, delimiter=',', dtype=float)
        x_all = torch.tensor(x_all, dtype=torch.float)
        num_node_features = x_all.shape[1]
    elif os.path.exists(node_labels_path):
        node_labels = np.loadtxt(node_labels_path, dtype=int)
        node_labels = torch.tensor(node_labels, dtype=torch.long).unsqueeze(1)
        num_node_features = int(node_labels.max()) + 1
        x_all = torch.nn.functional.one_hot(node_labels.squeeze(), num_classes=num_node_features).float()
    else:
        # fallback: one-hot degree vector
        degrees = np.zeros(graph_indicator.shape[0], dtype=int)
        for e in edges:
            degrees[e[0]] += 1
            degrees[e[1]] += 1
        num_node_features = int(degrees.max()) + 1
        x_all = torch.nn.functional.one_hot(torch.tensor(degrees), num_classes=num_node_features).float()

    # Build Data objects per graph
    dataset = []
    for g in range(num_graphs):
        node_mask = graph_indicator == g
        node_idx = np.where(node_mask)[0]
        idx_map = {old: new for new, old in enumerate(node_idx)}
        
        # Select edges for this graph
        mask_edges = [e for e in edges if e[0] in idx_map and e[1] in idx_map]
        if len(mask_edges) == 0:
            edge_index = torch.empty((2,0), dtype=torch.long)
        else:
            edge_index = torch.tensor([[idx_map[e[0]], idx_map[e[1]]] for e in mask_edges], dtype=torch.long).T
        
        x_graph = x_all[node_mask]
        y_graph = torch.tensor([graph_labels[g]], dtype=torch.long)
        
        dataset.append(Data(x=x_graph, edge_index=edge_index, y=y_graph))
    
    # Shuffle and split 80/20
    np.random.seed(12345)
    np.random.shuffle(dataset)
    train_size = int(0.8 * len(dataset))
    train_dataset = dataset[:train_size]
    test_dataset = dataset[train_size:]
    
    return dataset, train_dataset, test_dataset

# === LOAD DATA ===
dataset, train_dataset, test_dataset = get_dataset_from_files(DATASET_NAME, root="/kaggle/input/graph-dataset")
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Dataset: {DATASET_NAME}")
print(f"Number of graphs: {len(dataset)}")
print(f"Number of classes: {max([d.y.item() for d in dataset]) + 1}")
print(f"Number of node features: {dataset[0].x.shape[1]}")
print(f"Embedding dimensions: {HIDDEN_DIM}")
print("="*40)

# === GIN MODEL ===
class GIN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.15, num_layers=5):
        super().__init__()
        self.dropout = dropout
        self.num_layers = num_layers
        
        self.convs = torch.nn.ModuleList()
        for i in range(num_layers):
            input_dim = in_channels if i == 0 else hidden_channels
            mlp = Sequential(
                Linear(input_dim, hidden_channels),
                BatchNorm1d(hidden_channels),
                ReLU(),
                Linear(hidden_channels, hidden_channels),
                ReLU()
            )
            conv = GINConv(mlp)
            self.convs.append(conv)
        
        # Classifier
        self.lin1 = Linear(hidden_channels*num_layers, hidden_channels*num_layers)
        self.lin2 = Linear(hidden_channels*num_layers, out_channels)
    
    def forward(self, x, edge_index, batch):
        layer_embeddings = []
        for conv in self.convs:
            x = conv(x, edge_index)
            x = x.relu()
            x = F.dropout(x, p=self.dropout, training=self.training)
            layer_embeddings.append(x)
        
        pooled = [global_add_pool(h, batch) for h in layer_embeddings]
        graph_embedding = torch.cat(pooled, dim=1)
        
        h = self.lin1(graph_embedding)
        h = h.relu()
        h = F.dropout(h, p=0.5, training=self.training)
        out = self.lin2(h)
        
        return F.log_softmax(out, dim=1), F.softmax(out, dim=1), graph_embedding

# === SETUP MODEL ===
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GIN(dataset[0].x.shape[1], HIDDEN_DIM, max([d.y.item() for d in dataset]) + 1, DROPOUT).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# === TRAIN & TEST FUNCTIONS ===
def train():
    model.train()
    total_loss = 0
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        out, _, _ = model(data.x, data.edge_index, data.batch)
        loss = F.nll_loss(out, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

def test(loader):
    model.eval()
    all_preds, all_labels, all_probs, embeddings_list = [], [], [], []
    for data in loader:
        data = data.to(device)
        out, probs, embeddings = model(data.x, data.edge_index, data.batch)
        pred = probs.argmax(dim=1)
        all_preds.append(pred.detach().cpu())
        all_labels.append(data.y.detach().cpu())
        all_probs.append(probs.detach().cpu())
        embeddings_list.append(embeddings.detach().cpu())
    y_pred = torch.cat(all_preds).numpy()
    y_true = torch.cat(all_labels).numpy()
    y_probs = torch.cat(all_probs).numpy()
    final_embeddings = torch.cat(embeddings_list)
    return y_true, y_pred, y_probs, final_embeddings

# === TRAINING LOOP ===
start_train = time.time()
for epoch in range(1, EPOCHS+1):
    loss = train()
    if epoch % 10 == 0:
        y_true_monitor, y_pred_monitor, _, _ = test(test_loader)
        acc_monitor = accuracy_score(y_true_monitor, y_pred_monitor)
        print(f"Epoch {epoch:03d} | Loss {loss:.4f} | Test Acc {acc_monitor:.4f}")
end_train = time.time()

# === FINAL EVALUATION ===
y_true, y_pred, y_probs, final_embeddings = test(test_loader)
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='weighted')
try:
    auc = roc_auc_score(y_true, y_probs, multi_class='ovr')
except:
    auc = 0.0
memory_used = psutil.Process(os.getpid()).memory_info().rss / 1024**2

print("\n" + "="*40)
print("FINAL RESULTS")
print("="*40)
print(f"Dataset: {DATASET_NAME}")
print(f"Accuracy: {acc:.4f}")
print(f"F1-score: {f1:.4f}")
print(f"AUC: {auc:.4f}")
print(f"Training Time: {end_train-start_train:.2f} sec")
print(f"Memory Used: {memory_used:.2f} MB")
print("="*40)

**its the Bomb Devil**